# 05. Лучший текстовый подход + EfficientNet

Финальный обучающий эксперимент: сначала сравниваются `answer_only`, `answer_description` и `nlp`, затем лучший текстовый подход автоматически объединяется с признаками `effnet_feat_*`.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymorphy3

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

## Загрузка данных

In [ ]:
DATA_PATH = Path("../data_with_efficientnet_features.csv")
df = pd.read_csv(DATA_PATH)

effnet_cols = [col for col in df.columns if col.startswith("effnet_feat_")]
assert effnet_cols, "В датасете не найдены колонки effnet_feat_*"

print(f"Rows: {len(df)}")
print(f"EfficientNet features: {len(effnet_cols)}")
df[["id", "answer", "description", "difficulty"]].head()

## Общие функции

In [ ]:
RANDOM_STATE = 42


def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
VOWELS = set("аеёиоуыэюя")
RARE_LETTERS = set("фщъёцэ")
morph = pymorphy3.MorphAnalyzer()


def clean_and_lemmatize(text):
    if pd.isna(text):
        return "пусто"
    words = re.findall(r"[а-яёa-z]+", str(text).lower())
    if not words:
        return "пусто"
    return " ".join(morph.parse(word)[0].normal_form for word in words)


def extract_text_stats(text, prefix=""):
    if pd.isna(text):
        text = ""
    text = str(text).lower().strip()
    if "|" in text:
        text = text.split("|")[0].strip()

    compact = text.replace(" ", "")
    words = [word for word in text.split() if word]
    letters = [char for char in compact if char.isalpha()]
    cyrillic_letters = [char for char in letters if "а" <= char <= "я" or char == "ё"]
    latin_letters = [char for char in letters if "a" <= char <= "z"]

    len_chars = len(compact)
    vowel_count = sum(char in VOWELS for char in compact)
    consonant_count = sum(char.isalpha() and char not in VOWELS for char in compact)
    rare_count = sum(char in RARE_LETTERS for char in compact)
    unique_chars = len(set(compact))

    return pd.Series(
        {
            f"{prefix}len_chars": len_chars,
            f"{prefix}len_words": len(words),
            f"{prefix}avg_word_len": len_chars / len(words) if words else 0.0,
            f"{prefix}vowel_count": vowel_count,
            f"{prefix}consonant_count": consonant_count,
            f"{prefix}rare_count": rare_count,
            f"{prefix}unique_chars": unique_chars,
            f"{prefix}vowel_ratio": vowel_count / len_chars if len_chars else 0.0,
            f"{prefix}rare_ratio": rare_count / len_chars if len_chars else 0.0,
            f"{prefix}unique_ratio": unique_chars / len_chars if len_chars else 0.0,
            f"{prefix}cyrillic_ratio": len(cyrillic_letters) / len(letters) if letters else 0.0,
            f"{prefix}latin_ratio": len(latin_letters) / len(letters) if letters else 0.0,
            f"{prefix}has_dash": int("-" in text),
            f"{prefix}has_digits": int(any(char.isdigit() for char in text)),
        }
    )


def extract_rebus_logic(row):
    desc = str(row["description"]).lower()
    ans = str(row["answer"]).lower().split("|")[0]
    pos_prepositions = [" в ", " на ", " под ", " над ", " за ", " перед ", " из "]
    rebus_terms = ["запятая", "перевернутый", "вверх ногами", "зачеркнуть", "буква", "цифра"]

    return pd.Series(
        {
            "comma_count": desc.count(",") + desc.count("'"),
            "desc_len": len(desc),
            "ans_len": len(ans.replace(" ", "")),
            "has_pos_prep": int(any(prep in desc for prep in pos_prepositions)),
            "has_rebus_terms": int(any(term in desc for term in rebus_terms)),
            "is_multi_word_ans": int(" " in ans.strip()),
            "variant_count": str(row["answer"]).count("|") + 1,
        }
    )


def scaled_model(model):
    return Pipeline([("scaler", StandardScaler()), ("model", model)])


def numeric_models():
    return {
        "dummy_mean": DummyRegressor(strategy="mean"),
        "linear_regression": scaled_model(LinearRegression()),
        "ridge": scaled_model(Ridge(alpha=1.0)),
        "lasso": scaled_model(Lasso(alpha=0.001, max_iter=10_000, random_state=RANDOM_STATE)),
        "elastic_net": scaled_model(ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10_000, random_state=RANDOM_STATE)),
        "svr_rbf": scaled_model(SVR(kernel="rbf", C=1.0, epsilon=0.05)),
        "knn_5": scaled_model(KNeighborsRegressor(n_neighbors=5, weights="distance")),
        "random_forest": RandomForestRegressor(n_estimators=500, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
        "extra_trees": ExtraTreesRegressor(n_estimators=500, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
        "gradient_boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.03, max_depth=2, min_samples_leaf=5, random_state=RANDOM_STATE),
    }


def evaluate_models(approach_name, models, X, y):
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scoring = {"r2": "r2", "mae": "neg_mean_absolute_error", "rmse": make_scorer(rmse_score, greater_is_better=False)}
    rows = []
    for model_name, model in models.items():
        scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
        rows.append(
            {
                "approach": approach_name,
                "model": model_name,
                "r2_mean": scores["test_r2"].mean(),
                "r2_std": scores["test_r2"].std(),
                "mae_mean": -scores["test_mae"].mean(),
                "mae_std": scores["test_mae"].std(),
                "rmse_mean": -scores["test_rmse"].mean(),
                "rmse_std": scores["test_rmse"].std(),
            }
        )
    return pd.DataFrame(rows).sort_values("mae_mean").reset_index(drop=True)


## Подготовка трех текстовых подходов

In [ ]:
y = df["difficulty"].copy()

answer_features = df["answer"].apply(lambda value: extract_text_stats(value))
all_text_features = pd.concat(
    [
        df["answer"].apply(lambda value: extract_text_stats(value, prefix="ans_")),
        df["description"].apply(lambda value: extract_text_stats(value, prefix="desc_")),
    ],
    axis=1,
)

nlp_features = df.copy()
nlp_features["lemma_description"] = nlp_features["description"].apply(clean_and_lemmatize)
nlp_features["lemma_answer"] = nlp_features["answer"].apply(lambda value: clean_and_lemmatize(str(value).split("|")[0]))
rebus_features = nlp_features.apply(extract_rebus_logic, axis=1)
X_nlp = pd.concat([nlp_features[["lemma_description", "lemma_answer"]], rebus_features], axis=1)
nlp_numeric_cols = rebus_features.columns.tolist()

text_approaches = {
    "answer_only": {
        "X": answer_features,
        "models": numeric_models(),
        "type": "numeric",
    },
    "answer_description": {
        "X": all_text_features,
        "models": numeric_models(),
        "type": "numeric",
    },
    "nlp": {
        "X": X_nlp,
        "models": None,
        "type": "nlp",
    },
}

nlp_preprocessor = ColumnTransformer(
    transformers=[
        ("desc_tfidf", TfidfVectorizer(max_features=500), "lemma_description"),
        ("ans_tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 4), max_features=300), "lemma_answer"),
        ("num", StandardScaler(), nlp_numeric_cols),
    ]
)


def nlp_model(model):
    return Pipeline([("preprocessor", nlp_preprocessor), ("model", model)])


text_approaches["nlp"]["models"] = {
    "ridge": nlp_model(Ridge(alpha=1.0)),
    "lasso": nlp_model(Lasso(alpha=0.001, max_iter=10_000, random_state=RANDOM_STATE)),
    "elastic_net": nlp_model(ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10_000, random_state=RANDOM_STATE)),
    "gradient_boosting": nlp_model(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE)),
    "random_forest": nlp_model(RandomForestRegressor(n_estimators=300, max_depth=5, random_state=RANDOM_STATE)),
}

{key: value["X"].shape for key, value in text_approaches.items()}

## Сравнение текстовых подходов

In [ ]:
text_cv_metrics = pd.concat(
    [
        evaluate_models(approach_name, approach["models"], approach["X"], y)
        for approach_name, approach in text_approaches.items()
    ],
    ignore_index=True,
).sort_values("mae_mean").reset_index(drop=True)

best_text_row = text_cv_metrics.iloc[0]
best_text_approach = best_text_row["approach"]
best_text_model_name = best_text_row["model"]

print(f"Best text approach: {best_text_approach} / {best_text_model_name}")
text_cv_metrics

## Объединение лучшего текстового подхода с EfficientNet

In [ ]:
N_EFFNET_COMPONENTS = min(50, len(df) - 1, len(effnet_cols))


def make_combined_setup(best_approach_name):
    if best_approach_name == "answer_only":
        X_text = text_approaches["answer_only"]["X"]
        X_combined = pd.concat([X_text, df[effnet_cols]], axis=1)
        text_cols = X_text.columns.tolist()
        preprocessor = ColumnTransformer(
            transformers=[
                ("text", StandardScaler(), text_cols),
                (
                    "effnet",
                    Pipeline(
                        [
                            ("scaler", StandardScaler()),
                            ("pca", PCA(n_components=N_EFFNET_COMPONENTS, random_state=RANDOM_STATE)),
                        ]
                    ),
                    effnet_cols,
                ),
            ]
        )
        return X_combined, preprocessor

    if best_approach_name == "answer_description":
        X_text = text_approaches["answer_description"]["X"]
        X_combined = pd.concat([X_text, df[effnet_cols]], axis=1)
        text_cols = X_text.columns.tolist()
        preprocessor = ColumnTransformer(
            transformers=[
                ("text", StandardScaler(), text_cols),
                (
                    "effnet",
                    Pipeline(
                        [
                            ("scaler", StandardScaler()),
                            ("pca", PCA(n_components=N_EFFNET_COMPONENTS, random_state=RANDOM_STATE)),
                        ]
                    ),
                    effnet_cols,
                ),
            ]
        )
        return X_combined, preprocessor

    if best_approach_name == "nlp":
        X_combined = pd.concat([X_nlp, df[effnet_cols]], axis=1)
        preprocessor = ColumnTransformer(
            transformers=[
                ("desc_tfidf", TfidfVectorizer(max_features=500), "lemma_description"),
                ("ans_tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 4), max_features=300), "lemma_answer"),
                ("num", StandardScaler(), nlp_numeric_cols),
                (
                    "effnet",
                    Pipeline(
                        [
                            ("scaler", StandardScaler()),
                            ("pca", PCA(n_components=N_EFFNET_COMPONENTS, random_state=RANDOM_STATE)),
                        ]
                    ),
                    effnet_cols,
                ),
            ]
        )
        return X_combined, preprocessor

    raise ValueError(f"Unknown approach: {best_approach_name}")


X_combined, combined_preprocessor = make_combined_setup(best_text_approach)

for forbidden_col in ["answer", "description"]:
    assert forbidden_col not in X_combined.columns

X_combined.shape

## Обучение объединенной модели

In [ ]:
def combined_model(model):
    return Pipeline([("preprocessor", combined_preprocessor), ("model", model)])


combined_models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "linear_regression": combined_model(LinearRegression()),
    "ridge": combined_model(Ridge(alpha=10.0)),
    "elastic_net": combined_model(ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10_000, random_state=RANDOM_STATE)),
    "svr_rbf": combined_model(SVR(kernel="rbf", C=1.0, epsilon=0.05)),
    "knn_5": combined_model(KNeighborsRegressor(n_neighbors=5, weights="distance")),
    "random_forest": combined_model(RandomForestRegressor(n_estimators=500, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE)),
    "extra_trees": combined_model(ExtraTreesRegressor(n_estimators=500, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE)),
    "gradient_boosting": combined_model(GradientBoostingRegressor(n_estimators=200, learning_rate=0.03, max_depth=2, min_samples_leaf=5, random_state=RANDOM_STATE)),
}

combined_cv_metrics = evaluate_models(
    f"{best_text_approach}_plus_efficientnet",
    combined_models,
    X_combined,
    y,
)
combined_cv_metrics

## Holdout-проверка лучшей объединенной модели

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=RANDOM_STATE
)

best_combined_model_name = combined_cv_metrics.iloc[0]["model"]
best_combined_model = clone(combined_models[best_combined_model_name])
best_combined_model.fit(X_train, y_train)
y_pred = best_combined_model.predict(X_test)

holdout_metrics = pd.DataFrame(
    [
        {
            "approach": f"{best_text_approach}_plus_efficientnet",
            "model": best_combined_model_name,
            "mae": mean_absolute_error(y_test, y_pred),
            "r2": r2_score(y_test, y_pred),
        }
    ]
)
holdout_metrics

## Ошибки и график

In [ ]:
predictions = pd.DataFrame(
    {
        "answer": df.loc[y_test.index, "answer"],
        "description": df.loc[y_test.index, "description"],
        "actual_difficulty": y_test,
        "predicted_difficulty": y_pred,
        "absolute_error": np.abs(y_test - y_pred),
    }
).sort_values("absolute_error", ascending=False)

predictions.head(20)

In [ ]:
plot_df = pd.DataFrame({"Actual": y_test, "Predicted": y_pred}).sort_values("Actual").reset_index(drop=True)

plt.figure(figsize=(12, 6))
plt.plot(plot_df["Actual"].values, label="Actual", linewidth=2)
plt.plot(plot_df["Predicted"].values, label="Predicted", alpha=0.75)
plt.title(f"{best_text_approach} + EfficientNet: Actual vs Predicted ({best_combined_model_name})")
plt.xlabel("Sample index sorted by actual difficulty")
plt.ylabel("Difficulty")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Итоговое сравнение

In [ ]:
final_comparison = pd.concat(
    [
        text_cv_metrics.groupby("approach", as_index=False).first(),
        combined_cv_metrics.groupby("approach", as_index=False).first(),
    ],
    ignore_index=True,
).sort_values("mae_mean").reset_index(drop=True)

final_comparison